In [1]:
import requests
import pandas as pd
import json
import time
from urllib.parse import quote

In [2]:
BANK_NAME = "KDB"
BANK_CODE = "KDB"

INIT_URL = "https://banking.kdb.co.kr/bp/BZCEEI08REA.jct"
DATA_URL = "https://banking.kdb.co.kr/bp/CBADIE06R01.jct"
REFERER_URL = "https://banking.kdb.co.kr/bp/CBADIE06N01.act?utm_source=chatgpt.com"

COMMON_HEADERS = {
    "accept": "*/*",
    "content-type": "application/x-www-form-urlencoded",
    "origin": "https://banking.kdb.co.kr",
    "referer": REFERER_URL,
    "user-agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/146.0.0.0 Safari/537.36"
    ),
    "x-requested-with": "XMLHttpRequest",
}

In [3]:
def generate_month_end_dates(start_date="2004-01-31", end_date="2019-12-31"):
    """
    월말 기준 날짜 문자열(YYYYMMDD) 리스트 생성
    """
    dates = []
    current = pd.to_datetime(start_date) + pd.offsets.MonthEnd(0)
    end = pd.to_datetime(end_date)

    while current <= end:
        dates.append(current.strftime("%Y%m%d"))
        current = current + pd.offsets.MonthEnd(1)

    return dates

In [4]:
def init_kdb_session(session, target_yyyymmdd):
    """
    첫 번째 요청:
    https://banking.kdb.co.kr/bp/BZCEEI08REA.jct

    body 형식:
    _JSON_ = {"_SECT_PGM_INSTALL_C_":"2","ISO_NTN_SYM_C":"KO","BSE_DT":"YYYYMMDD"}
    """
    payload_obj = {
        "_SECT_PGM_INSTALL_C_": "2",
        "ISO_NTN_SYM_C": "KO",
        "BSE_DT": target_yyyymmdd
    }

    resp = session.post(
        INIT_URL,
        headers=COMMON_HEADERS,
        data={"_JSON_": json.dumps(payload_obj, ensure_ascii=False)},
        timeout=30
    )
    resp.raise_for_status()
    return resp.json()

In [5]:
def fetch_kdb_json(session, target_yyyymmdd, debug=False):
    """
    두 번째 요청:
    https://banking.kdb.co.kr/bp/CBADIE06R01.jct

    중요:
    body가 data={"_JSON_": ...} 가 아니라
    URL-encoded JSON 문자열 자체임.
    """
    payload_obj = {
        "_SECT_PGM_INSTALL_C_": "2",
        "BSE_DT": target_yyyymmdd
    }

    encoded_body = quote(json.dumps(payload_obj, ensure_ascii=False))

    resp = session.post(
        DATA_URL,
        headers=COMMON_HEADERS,
        data=encoded_body,
        timeout=60
    )
    resp.raise_for_status()

    if debug:
        print("status:", resp.status_code)
        print("content-type:", resp.headers.get("content-type"))
        print("text first 1000 chars:")
        print(resp.text[:1000])

    return resp.json()

In [6]:
def normalize_kdb_rows(raw_json, target_yyyymmdd):
    """
    KDB 응답을 예시 엑셀 형식으로 변환

    출력 열:
    bank, bank_code, target_date, currency, maturity, rate, product
    """

    # 실제 데이터는 raw_json["REC"] 안에 있음
    if isinstance(raw_json, dict):
        data_list = raw_json.get("REC", [])
    elif isinstance(raw_json, list):
        data_list = raw_json
    else:
        data_list = []

    if not isinstance(data_list, list) or len(data_list) == 0:
        return pd.DataFrame(columns=[
            "bank", "bank_code", "target_date",
            "currency", "maturity", "rate", "product"
        ])

    target_date_fmt = pd.to_datetime(
        target_yyyymmdd, format="%Y%m%d"
    ).strftime("%Y-%m-%d")

    rows = []
    for item in data_list:
        rows.append({
            "bank": BANK_NAME,
            "bank_code": BANK_CODE,
            "target_date": target_date_fmt,
            "currency": item.get("CUR_C"),
            "maturity": item.get("PRD_IRT_STG_NM1") or item.get("STG_ABV_NM1"),
            "rate": pd.to_numeric(item.get("IRT_BSE_VL"), errors="coerce"),
            "product": item.get("MNG_ITSR_DES") or item.get("ITSR_DES") or item.get("MNG_ITSR_GRP_NM")
        })

    df = pd.DataFrame(rows)

    # 완전 중복 제거
    df = df.drop_duplicates().reset_index(drop=True)

    return df

In [7]:
session = requests.Session()
test_date = "20260330"

init_result = init_kdb_session(session, test_date)
print("init_result:", init_result)

raw_json = fetch_kdb_json(session, test_date, debug=True)
print("json type:", type(raw_json))
if isinstance(raw_json, dict):
    print("json keys:", list(raw_json.keys())[:20])

df_test = normalize_kdb_rows(raw_json, test_date)

print("rows:", len(df_test))
print(df_test.head(10))

init_result: {'BNK_HDY_YN': 'N', 'COMMON_HEAD': {'MESSAGE': '', 'CODE': '', 'ERROR': False}}
status: 200
content-type: application/json; charset=UTF-8
text first 1000 chars:
{"REC":[{"PRD_IRT_STG_NM1":"1주일물","PFR_RKG_VL":"0","AVR_ITSR":"0","LST_CHG_DTM":"2026-03-30 00:20:00.0","PRV_CRP_TC":"00","PRD_IRT_STG_TC1":"0000007","DEL_YN":"N","PRD_IRT_STG_TC2":"0","PRD_IRT_STG_TC3":"0","ALY_STT_DT":"20260330","PRD_IRT_C":"600020010001","CHG_DTT_YN":"N","MNG_ITSR_GRP_NM":"외화보통예금","CPRE_MANR_C3":"00","MNG_ITSR_DES":"외화보통예금LIBOR","GUID":"ITB20190513045539839BZC_MS210000001913","STG_ABV_NM1":"1주일물","CPRE_MANR_C1":"01","IRT_GRP_NM":"외화보통예금","CPRE_MANR_C2":"00","ITSR_DES":"외화보통예금LIBOR","GUID_PRG_SNO":"2756","LST_CHG_USID":"BATCH","IRT_BSE_VL":"0.000000","IRT_KD_C":"5003","CUR_C":"USD","IRT_PRD_GRP_C":"00"},{"PRD_IRT_STG_NM1":"1주일물","PFR_RKG_VL":"0","AVR_ITSR":"0","LST_CHG_DTM":"2026-03-30 00:20:00.0","PRV_CRP_TC":"00","PRD_IRT_STG_TC1":"0000007","DEL_YN":"N","PRD_IRT_STG_TC2":"0","PRD_IRT_STG_TC3":"

In [8]:
def crawl_kdb_range(start_date="2004-01-31", end_date="2019-12-31", sleep_sec=0.2):
    """
    월말 기준으로 날짜를 순회하면서 데이터 수집
    """
    session = requests.Session()
    all_frames = []
    failed_dates = []

    date_list = generate_month_end_dates(start_date, end_date)
    print(f"총 {len(date_list)}개 날짜 수집 시작")

    for i, yyyymmdd in enumerate(date_list, 1):
        try:
            # 1차 초기화 요청
            init_kdb_session(session, yyyymmdd)

            # 2차 실제 데이터 요청
            raw_json = fetch_kdb_json(session, yyyymmdd, debug=False)

            # 정리
            df_day = normalize_kdb_rows(raw_json, yyyymmdd)

            if not df_day.empty:
                all_frames.append(df_day)

            print(f"[{i}/{len(date_list)}] {yyyymmdd} 완료 - {len(df_day)} rows")

        except Exception as e:
            print(f"[{i}/{len(date_list)}] {yyyymmdd} 실패 - {e}")
            failed_dates.append({
                "target_date": yyyymmdd,
                "error": str(e)
            })

        time.sleep(sleep_sec)

    if all_frames:
        final_df = pd.concat(all_frames, ignore_index=True)
    else:
        final_df = pd.DataFrame(columns=[
            "bank", "bank_code", "target_date",
            "currency", "maturity", "rate", "product"
        ])

    failed_df = pd.DataFrame(failed_dates)

    return final_df, failed_df

In [9]:
kdb_df, failed_df = crawl_kdb_range(
    start_date="2004-01-31",
    end_date="2019-12-31",
    sleep_sec=0.2
)

print("\n최종 shape:", kdb_df.shape)
print(kdb_df.head())
print("\n실패 건수:", len(failed_df))
print(failed_df.head())

총 192개 날짜 수집 시작
[1/192] 20040131 완료 - 459 rows
[2/192] 20040229 완료 - 459 rows
[3/192] 20040331 완료 - 459 rows
[4/192] 20040430 완료 - 459 rows
[5/192] 20040531 완료 - 459 rows
[6/192] 20040630 완료 - 459 rows
[7/192] 20040731 완료 - 459 rows
[8/192] 20040831 완료 - 459 rows
[9/192] 20040930 완료 - 459 rows
[10/192] 20041031 완료 - 459 rows
[11/192] 20041130 완료 - 459 rows
[12/192] 20041231 완료 - 459 rows
[13/192] 20050131 완료 - 459 rows
[14/192] 20050228 완료 - 459 rows
[15/192] 20050331 완료 - 459 rows
[16/192] 20050430 완료 - 459 rows
[17/192] 20050531 완료 - 459 rows
[18/192] 20050630 완료 - 459 rows
[19/192] 20050731 완료 - 459 rows
[20/192] 20050831 완료 - 459 rows
[21/192] 20050930 완료 - 459 rows
[22/192] 20051031 완료 - 459 rows
[23/192] 20051130 완료 - 459 rows
[24/192] 20051231 완료 - 459 rows
[25/192] 20060131 완료 - 459 rows
[26/192] 20060228 완료 - 459 rows
[27/192] 20060331 완료 - 459 rows
[28/192] 20060430 완료 - 459 rows
[29/192] 20060531 완료 - 459 rows
[30/192] 20060630 완료 - 459 rows
[31/192] 20060731 완료 - 459 rows
[

In [10]:
print("고유 날짜 수:", kdb_df["target_date"].nunique())
print("\n날짜별 행 수 상위:")
print(kdb_df["target_date"].value_counts().head(10))

print("\n샘플 날짜 확인:")
sample_dates = sorted(kdb_df["target_date"].dropna().unique())[:3] + sorted(kdb_df["target_date"].dropna().unique())[-3:]
for d in sample_dates:
    print("\n====", d, "====")
    print(kdb_df[kdb_df["target_date"] == d].head(5))

고유 날짜 수: 192

날짜별 행 수 상위:
target_date
2004-01-31    459
2004-02-29    459
2004-03-31    459
2004-04-30    459
2004-05-31    459
2004-06-30    459
2004-07-31    459
2004-08-31    459
2004-09-30    459
2004-10-31    459
Name: count, dtype: int64

샘플 날짜 확인:

==== 2004-01-31 ====
  bank bank_code target_date currency maturity  rate      product
0  KDB       KDB  2004-01-31      USD     1주일물   0.0  외화보통예금LIBOR
1  KDB       KDB  2004-01-31      USD     1주일물   0.0  외화보통예금LIBID
2  KDB       KDB  2004-01-31      CNH     1주일물   0.1     외화보통예금기준
3  KDB       KDB  2004-01-31      CHF     1주일물   0.0  외화보통예금LIBID
4  KDB       KDB  2004-01-31      CHF     1주일물   0.0     외화보통예금기준

==== 2004-02-29 ====
    bank bank_code target_date currency maturity  rate      product
459  KDB       KDB  2004-02-29      USD     1주일물   0.0  외화보통예금LIBOR
460  KDB       KDB  2004-02-29      USD     1주일물   0.0  외화보통예금LIBID
461  KDB       KDB  2004-02-29      CNH     1주일물   0.1     외화보통예금기준
462  KDB       KDB  2004-02-29   

In [11]:
output_file = "kdb_2004_2019_full.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    kdb_df.to_excel(writer, sheet_name="Sheet1", index=False)
    failed_df.to_excel(writer, sheet_name="failed_dates", index=False)

print(f"저장 완료: {output_file}")

저장 완료: kdb_2004_2019_full.xlsx
